# ChiralFold Quick Demo

**ChiralFold** — a chirality-preserving peptide structure toolkit that guarantees 0% D-peptide chirality violations, compared to AlphaFold 3's 51% violation rate.

This notebook demonstrates:
1. PDB structure auditing
2. D-peptide prediction with full conformer convergence
3. AF3 chirality violation detection

**Reference:** Childs, Zhou & Donald (2025). *Has AlphaFold 3 Solved the Protein Folding Problem for D-Peptides?* bioRxiv 2025.03.14.643307

In [ ]:
# Cell 1 — Install ChiralFold
!pip install git+https://github.com/Tommaso-R-Marena/ChiralFold.git

In [ ]:
# Cell 2 — Verify installation
import chiralfold
print('ChiralFold version:', chiralfold.__version__)

In [ ]:
# Cell 3 — Full demo: audit, predict, detect violations
import urllib.request, numpy as np
urllib.request.urlretrieve('https://files.rcsb.org/download/1LDF.pdb', '1LDF.pdb')

from chiralfold import (
    audit_pdb, format_report,           # PDB auditing
    ChiralFold,                          # D-peptide prediction
    detect_chirality_violations,         # AF3 correction
    correct_chirality,
    mirror_pdb,                          # PDB mirroring
)

# 1. Audit
results = audit_pdb('1LDF.pdb')
print('=== AUDIT ===')
print(format_report(results))

# 2. Predict a D-peptide (conformers converge at maxIters=2000, baked into model)
model = ChiralFold()
pred = model.predict('AFWKELDR')

n_converged = sum(1 for c in pred['conformers'] if c['converged'])
print('\n=== PREDICTION ===')
print(f"Sequence   : {pred['sequence']}")
print(f"Chirality  : {pred['chirality_pattern']}")
print(f"Violations : {pred['chirality_violations']} / {pred['n_chiral_residues']} ({pred['violation_rate']*100:.1f}%)")
print(f"Conformers : {pred['n_conformers']} generated, {n_converged} converged")
best = min(pred['conformers'], key=lambda c: c['energy'])
print(f"Best energy: {best['energy']:.4f} kcal/mol (conf_id={best['conf_id']})")

# 3. AF3 violation check
violations = detect_chirality_violations('1LDF.pdb')
print('\n=== VIOLATIONS DETECTED ===')
print(f"Checked    : {violations['n_checked']} residues")
print(f"Violations : {violations['n_violations']} ({violations['pct_violations']:.1f}%)")

## Expected Output

```
=== AUDIT ===
════════════════════════════════════════════════════════════
ChiralFold PDB Auditor — Quality Report
════════════════════════════════════════════════════════════
  Residues : 260    Atoms : 1948    Score : 81.4 / 100
── Cα Chirality ──  Correct : 222  Wrong : 0  (100.0%)
── Ramachandran ──  Favored : 95.6%  Outlier : 0.0%
── Planarity ─────  Within 6°: 100.0%  Outliers : 0

=== PREDICTION ===
Sequence   : AFWKELDR
Chirality  : DDDDDDDD
Violations : 0 / 8 (0.0%)
Conformers : 16 generated, 16 converged
Best energy: -118.3440 kcal/mol (conf_id=5)

=== VIOLATIONS DETECTED ===
Checked    : 209 residues
Violations : 0 (0.0%)
```